# Adult Income: responsible classification

**Question:** Can demographic and work-history data predict whether income exceeds $50K, while explicitly auditing group-level error rates?

Source: [UCI Adult](https://archive.ics.uci.edu/dataset/2/adult). This is an educational exercise—not a system for employment, credit, or benefits decisions.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

columns = ['age','workclass','fnlwgt','education','education_num','marital_status','occupation','relationship','race','sex','capital_gain','capital_loss','hours_per_week','native_country','income']
df = pd.read_csv(Path('../datasets/portfolio-expansion/adult/adult.data'), names=columns, na_values=' ?', skipinitialspace=True)
y = df.pop('income').str.contains('>50K').astype(int)
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=.2, stratify=y, random_state=42)
num = X_train.select_dtypes('number').columns; cat = X_train.select_dtypes(exclude='number').columns
prep = ColumnTransformer([('num', Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]), num), ('cat', Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), cat)])
model = Pipeline([('prep',prep),('model',LogisticRegression(max_iter=2000, class_weight='balanced'))]).fit(X_train,y_train)
prediction = model.predict(X_test)
print(classification_report(y_test,prediction))

              precision    recall  f1-score   support

           0       0.94      0.80      0.87      4945
           1       0.57      0.85      0.69      1568

    accuracy                           0.81      6513
   macro avg       0.76      0.83      0.78      6513
weighted avg       0.86      0.81      0.82      6513



In [2]:
# Audit—not justify—differences in error rates across groups.
audit = X_test[['sex','race']].copy(); audit['actual']=y_test; audit['predicted']=prediction
report = audit.groupby(['sex','race']).apply(lambda g: pd.Series({'n':len(g),'selection_rate':g.predicted.mean(),'true_positive_rate':g.loc[g.actual.eq(1),'predicted'].mean()}), include_groups=False)
display(report.sort_values('n',ascending=False).round(3))
# Small groups have unstable estimates. Any real use needs governance, consent, and a domain-specific fairness review.

n  selection_rate  true_positive_rate
sex    race                                                          
Male   White               3807.0           0.484               0.879
Female White               1726.0           0.155               0.780
Male   Black                333.0           0.264               0.760
Female Black                329.0           0.091               0.722
Male   Asian-Pac-Islander   136.0           0.507               0.818
Female Asian-Pac-Islander    64.0           0.172               0.625
Male   Amer-Indian-Eskimo    45.0           0.222               0.800
       Other                 34.0           0.206               1.000
Female Amer-Indian-Eskimo    28.0           0.000               0.000
       Other                 11.0           0.182               1.000

## Portfolio takeaway

This project demonstrates that a good data scientist evaluates societal impact as well as aggregate accuracy. Future work: calibrated probabilities, confidence intervals for group metrics, and a documented decision-policy review.